# Model 3 — Gated Segmentation Network
### CS 6140 — ML Final Project
**1:** EfficientNet-B0 Gate Classifier  
**2:** ResNet-34 / UNet Segmenter

## Install Dependencies

In [1]:
!pip install segmentation-models-pytorch albumentations tqdm -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01
Dependencies installed.


## Imports

In [2]:
import os, time, logging
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, ConfusionMatrixDisplay
)
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s')
log = logging.getLogger(__name__)
print('Imports done.')

Imports done.


## Paths & Shared Config

In [ ]:
# Setting up paths and constants
DATA_DIR = Path("/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data")
AUTHENTIC_DIR = DATA_DIR / 'train_images' / 'authentic'
FORGED_DIR    = DATA_DIR / 'train_images' / 'forged'
MASK_DIR      = DATA_DIR / 'train_masks'
SUPP_AUTH     = DATA_DIR / 'supplemental_images' / 'authentic'
SUPP_FORG     = DATA_DIR / 'supplemental_images' / 'forged'
OUTPUT_DIR    = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

VALID_EXT   = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
RANDOM_SEED = 42
IMAGE_SIZE  = 512

# Device
def get_device():
    if torch.cuda.is_available():
        log.info('Using device: CUDA'); return torch.device('cuda')
    log.info('Using device: CPU');  return torch.device('cpu')

DEVICE = get_device()

# Verify paths
print('DATA_DIR exists:', DATA_DIR.exists())
print('AUTHENTIC_DIR  :', AUTHENTIC_DIR.exists())
print('FORGED_DIR     :', FORGED_DIR.exists())
print('MASK_DIR       :', MASK_DIR.exists())

2026-04-21 20:18:13,216  Using device: CUDA


DATA_DIR exists: True
AUTHENTIC_DIR  : True
FORGED_DIR     : True
MASK_DIR       : True


## Shared Utilities (dataset loader, transforms)

In [4]:
def collect_pairs(authentic_dir, forged_dir, samples_per_class=None, seed=RANDOM_SEED):
    auth_files = sorted(f for f in authentic_dir.glob('*') if f.suffix.lower() in VALID_EXT)
    forg_files = sorted(f for f in forged_dir.glob('*')    if f.suffix.lower() in VALID_EXT)
    if SUPP_AUTH.exists():
        auth_files += sorted(f for f in SUPP_AUTH.glob('*') if f.suffix.lower() in VALID_EXT)
    if SUPP_FORG.exists():
        forg_files += sorted(f for f in SUPP_FORG.glob('*') if f.suffix.lower() in VALID_EXT)
    log.info(f'Authentic: {len(auth_files)}  |  Forged: {len(forg_files)}')
    rng = np.random.default_rng(seed)
    if samples_per_class:
        auth_files = list(rng.choice(auth_files, size=min(samples_per_class, len(auth_files)), replace=False))
        forg_files = list(rng.choice(forg_files, size=min(samples_per_class, len(forg_files)), replace=False))
    pairs = [(p, 0) for p in auth_files] + [(p, 1) for p in forg_files]
    rng.shuffle(pairs)
    return pairs


def split_pairs(pairs, seed=RANDOM_SEED):
    labels = [p[1] for p in pairs]
    train, temp = train_test_split(pairs, test_size=0.30, stratify=labels, random_state=seed)
    temp_labels  = [p[1] for p in temp]
    val, test    = train_test_split(temp,  test_size=0.50, stratify=temp_labels, random_state=seed)
    log.info(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')
    return train, val, test


def save_checkpoint(state, path):
    torch.save(state, path)
    log.info(f'Checkpoint saved → {path}')


def plot_confusion_matrix(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Authentic', 'Forged']).plot(ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150); plt.close()
    log.info(f'Confusion matrix saved → {path}')


print('Shared utilities loaded.')

Shared utilities loaded.


---
# Phase 1 — Gate Classifier (EfficientNet-B0)

## Gate Config

In [ ]:
# Constants for Gate classifier
GATE_RESUME           = False   
GATE_SAMPLES_PER_CLASS = 1200
GATE_BATCH_SIZE        = 32
GATE_NUM_EPOCHS        = 20
GATE_LR                = 1e-4
GATE_WEIGHT_DECAY      = 1e-4
GATE_PATIENCE          = 5

GATE_CHECKPOINT = OUTPUT_DIR / 'gate_checkpoint.pth'
GATE_BEST       = OUTPUT_DIR / 'gate_best.pth'
print('Gate config set.')

Gate config set.


## Gate Dataset & Model

In [6]:
class GateDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs     = pairs
        self.transform = transform

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


def gate_transforms(train):
    ops = [transforms.ToPILImage()]
    if train:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ]
    return transforms.Compose(ops)


def build_gate_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_f, 1))
    return model


print('Gate dataset and model defined.')

Gate dataset and model defined.


## Gate Train / Eval Functions

In [7]:
def gate_train_epoch(model, loader, optimizer, criterion, device, epoch, total):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'Epoch [{epoch+1:02d}/{total}] Train', ncols=100)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(model(imgs)) >= 0.5).float()
        correct += (preds == labels).sum().item()
        n += imgs.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/n:.4f}'})
    return total_loss / n, correct / n


@torch.no_grad()
def gate_evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        probs  = torch.sigmoid(logits).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_preds.extend((probs >= 0.5).astype(int))
        all_labels.extend(labels.cpu().numpy().flatten().astype(int))
    n   = len(all_labels)
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    return total_loss / n, acc, f1, auc, all_preds, all_labels


print('Gate train/eval functions defined.')

Gate train/eval functions defined.


## Training the Gate Model

In [ ]:
# Data 
gate_pairs = collect_pairs(AUTHENTIC_DIR, FORGED_DIR, GATE_SAMPLES_PER_CLASS)
train_pairs, val_pairs, test_pairs = split_pairs(gate_pairs)

train_dl = DataLoader(GateDataset(train_pairs, gate_transforms(True)),
                      batch_size=GATE_BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(GateDataset(val_pairs,   gate_transforms(False)),
                      batch_size=GATE_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(GateDataset(test_pairs,  gate_transforms(False)),
                      batch_size=GATE_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Model 
gate_model     = build_gate_model().to(DEVICE)
gate_criterion = nn.BCEWithLogitsLoss()
gate_optimizer = AdamW(gate_model.parameters(), lr=GATE_LR, weight_decay=GATE_WEIGHT_DECAY)
gate_scheduler = CosineAnnealingLR(gate_optimizer, T_max=GATE_NUM_EPOCHS, eta_min=1e-6)

start_epoch, best_val_f1, epochs_no_improve = 0, 0.0, 0
train_losses, val_losses, val_f1s = [], [], []

if GATE_RESUME and GATE_CHECKPOINT.exists():
    ckpt = torch.load(GATE_CHECKPOINT, map_location='cpu')
    gate_model.load_state_dict(ckpt['model_state'])
    gate_optimizer.load_state_dict(ckpt['optimizer_state'])
    gate_scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch']
    best_val_f1   = ckpt['best_val_f1']
    log.info(f'Resumed from epoch {start_epoch}  (best val F1: {best_val_f1:.4f})')
elif GATE_RESUME:
    log.warning('GATE_RESUME=True but no checkpoint found. Training from scratch.')

# Training loop 
log.info('Starting gate training...')
for epoch in range(start_epoch, GATE_NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = gate_train_epoch(
        gate_model, train_dl, gate_optimizer, gate_criterion, DEVICE, epoch, GATE_NUM_EPOCHS)
    val_loss, val_acc, val_f1, val_auc, _, _ = gate_evaluate(
        gate_model, val_dl, gate_criterion, DEVICE)
    gate_scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    log.info(
        f'Epoch [{epoch+1:02d}/{GATE_NUM_EPOCHS}]  '
        f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  '
        f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  '
        f'val_F1={val_f1:.4f}  val_AUC={val_auc:.4f}  '
        f'({time.time()-t0:.1f}s)'
    )

    save_checkpoint({
        'epoch': epoch+1, 'model_state': gate_model.state_dict(),
        'optimizer_state': gate_optimizer.state_dict(),
        'scheduler_state': gate_scheduler.state_dict(),
        'best_val_f1': best_val_f1,
    }, GATE_CHECKPOINT)

    if val_f1 > best_val_f1:
        best_val_f1, epochs_no_improve = val_f1, 0
        torch.save(gate_model.state_dict(), GATE_BEST)
        log.info(f'  ↑ New best val F1: {best_val_f1:.4f} → {GATE_BEST}')
    else:
        epochs_no_improve += 1
        log.info(f'  No improvement ({epochs_no_improve}/{GATE_PATIENCE})')

    if epochs_no_improve >= GATE_PATIENCE:
        log.info(f'Early stopping — no improvement for {GATE_PATIENCE} epochs')
        break

log.info('Gate training complete.')

2026-04-21 20:18:27,278  Authentic: 2377  |  Forged: 2751
2026-04-21 20:18:27,297  Train: 1680  Val: 360  Test: 360


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s] 
2026-04-21 20:18:28,050  Starting gate training …
Epoch [01/20] Train: 100%|█████████████████| 53/53 [00:57<00:00,  1.09s/it, loss=0.6944, acc=0.5363]
2026-04-21 20:19:35,057  Epoch [01/20]  train_loss=0.7010  train_acc=0.5363  val_loss=0.6967  val_acc=0.5028  val_F1=0.5445  val_AUC=0.5173  (67.0s)
2026-04-21 20:19:35,180  Checkpoint saved → /kaggle/working/gate_checkpoint.pth
2026-04-21 20:19:35,224    ↑ New best val F1: 0.5445 → /kaggle/working/gate_best.pth
Epoch [02/20] Train: 100%|█████████████████| 53/53 [00:54<00:00,  1.02s/it, loss=0.7277, acc=0.5673]
2026-04-21 20:20:36,996  Epoch [02/20]  train_loss=0.6852  train_acc=0.5673  val_loss=0.6999  val_acc=0.5028  val_F1=0.5536  val_AUC=0.5040  (61.8s)
2026-04-21 20:20:37,157  Checkpoint saved → /kaggle/working/gate_checkpoint.pth
2026-04-21 20:20:37,211    ↑ New best val F1: 0.5536 → /kaggle/working/gate_best.pth
Epoch [03/20] Train: 100%|█████████████████| 53/53 [00:58<00:00,  1

## Gate Evaluation & Plots

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses,   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Gate — Loss Curves'); ax1.legend()
ax2.plot(val_f1s, label='Val F1', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1')
ax2.set_title('Gate — Validation F1'); ax2.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gate_training_curves.png', dpi=150)
plt.show()

# Test evaluation
gate_model.load_state_dict(torch.load(GATE_BEST, map_location=DEVICE))
_, test_acc, test_f1, test_auc, test_preds, test_labels = gate_evaluate(
    gate_model, test_dl, gate_criterion, DEVICE)

print('GATE — FINAL TEST SET RESULTS:\n')

print(f'  Accuracy : {test_acc:.4f}')
print(f'  F1       : {test_f1:.4f}')
print(f'  ROC-AUC  : {test_auc:.4f}')
print(classification_report(test_labels, test_preds, target_names=['Authentic','Forged']))

plot_confusion_matrix(
    test_labels, test_preds,
    title='Confusion Matrix — Gate (EfficientNet-B0) — Test Set',
    path=OUTPUT_DIR / 'gate_confusion_matrix.png'
)
plt.figure()
img_cm = plt.imread(str(OUTPUT_DIR / 'gate_confusion_matrix.png'))
plt.imshow(img_cm); plt.axis('off'); plt.show()

2026-04-21 21:07:15,913  Confusion matrix saved → /kaggle/working/gate_confusion_matrix.png



GATE — FINAL TEST SET RESULTS
  Accuracy : 0.7167
  F1       : 0.7287
  ROC-AUC  : 0.8326
              precision    recall  f1-score   support

   Authentic       0.74      0.67      0.70       180
      Forged       0.70      0.76      0.73       180

    accuracy                           0.72       360
   macro avg       0.72      0.72      0.72       360
weighted avg       0.72      0.72      0.72       360



---
# Phase 2 — Segmenter (ResNet-34 / UNet)

## Segmenter Config

In [10]:
# Constants for segmenter
SEG_RESUME       = False
SEG_BATCH_SIZE   = 16
SEG_NUM_EPOCHS   = 30
SEG_LR           = 3e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_PATIENCE     = 7
BCE_WEIGHT       = 0.5
DICE_WEIGHT      = 0.5

SEG_CHECKPOINT = OUTPUT_DIR / 'segmenter_checkpoint.pth'
SEG_BEST       = OUTPUT_DIR / 'segmenter_best.pth'
print('Segmenter config set.')

Segmenter config set.


## Segmenter Dataset, Model & Loss

In [24]:
class SegDataset(Dataset):
    def __init__(self, pairs, mask_dir, transform=None, image_size=512):
        self.pairs      = pairs
        self.mask_dir   = mask_dir
        self.transform  = transform
        self.image_size = image_size   

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, _ = self.pairs[idx]
        img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img  = cv2.resize(img, (self.image_size, self.image_size))
    
        mask_npy = self.mask_dir / (img_path.stem + '.npy')
        mask_png = self.mask_dir / (img_path.stem + '.png')
    
        if mask_npy.exists():
            mask = np.load(str(mask_npy))
            mask = np.squeeze(mask)              # (1, H, W) to (H, W)
            mask = mask.astype(np.float32)       # uint8 to float32 after squeeze
        elif mask_png.exists():
            mask = (cv2.imread(str(mask_png), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        else:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)

        # Mask dimension checks
        if mask.ndim != 2:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)
        if mask.shape[0] == 0 or mask.shape[1] == 0:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)
    
        # Squeeze out any extra dimensions (e.g. shape (H,W,1) or (1,H,W))
        mask = np.squeeze(mask)
    
        # If mask somehow ended up 1-D or 0-D, replace with zeros
        if mask.ndim != 2:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)
    
        # If mask is empty or has zero dimension, replace with zeros
        if mask.shape[0] == 0 or mask.shape[1] == 0:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)
    
        mask = cv2.resize(mask, (self.image_size, self.image_size),
                          interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0.5).astype(np.float32)
    
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask'].unsqueeze(0)
        else:
            img  = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)
        return img, mask


def seg_transforms(train):
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5), A.GaussNoise(p=0.3),
            A.Blur(blur_limit=3, p=0.2),
            A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


def build_segmenter():
    return smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=3, classes=1, activation=None,
        decoder_attention_type='scse',
    )


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_w=BCE_WEIGHT, dice_w=DICE_WEIGHT, smooth=1e-6):
        super().__init__()
        self.bce_w  = bce_w
        self.dice_w = dice_w
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        flat_p = probs.view(-1); flat_t = targets.view(-1)
        inter  = (flat_p * flat_t).sum()
        dice   = 1 - (2*inter + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)
        return self.bce_w * self.bce(logits, targets) + self.dice_w * dice


def seg_metrics(logits, targets, thresh=0.5):
    preds  = (torch.sigmoid(logits) >= thresh).float()
    smooth = 1e-6
    inter  = (preds * targets).sum()
    union  = preds.sum() + targets.sum()
    tp     = inter
    fp     = (preds * (1 - targets)).sum()
    fn     = ((1 - preds) * targets).sum()
    return {
        'dice':      ((2*inter + smooth) / (union + smooth)).item(),
        'iou':       ((inter + smooth)   / (union - inter + smooth)).item(),
        'precision': ((tp + smooth) / (tp + fp + smooth)).item(),
        'recall':    ((tp + smooth) / (tp + fn + smooth)).item(),
    }


print('Segmenter dataset, model and loss defined.')

Segmenter dataset, model and loss defined.


## Segmenter Train / Eval Functions

In [25]:
def seg_train_epoch(model, loader, optimizer, criterion, device, epoch, total):
    model.train()
    total_loss = 0.0
    metrics    = {'dice':0,'iou':0,'precision':0,'recall':0}
    pbar = tqdm(loader, desc=f'Epoch [{epoch+1:02d}/{total}] Train', ncols=110)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        m = seg_metrics(logits.detach(), masks)
        for k in metrics: metrics[k] += m[k] * imgs.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{m["dice"]:.4f}'})
    n = len(loader.dataset)
    return total_loss/n, {k: v/n for k,v in metrics.items()}


@torch.no_grad()
def seg_evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    metrics    = {'dice':0,'iou':0,'precision':0,'recall':0}
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item() * imgs.size(0)
        m = seg_metrics(logits, masks)
        for k in metrics: metrics[k] += m[k] * imgs.size(0)
    n = len(loader.dataset)
    return total_loss/n, {k: v/n for k,v in metrics.items()}


print('Segmenter train/eval functions defined.')

Segmenter train/eval functions defined.


In [22]:
# Debug — inspect a few mask files
import random
mask_files = list(MASK_DIR.glob('*.npy'))[:5]
for mf in mask_files:
    m = np.load(str(mf))
    print(f"{mf.name}  shape={m.shape}  dtype={m.dtype}  "
          f"min={m.min():.3f}  max={m.max():.3f}")

59069.npy  shape=(1, 512, 711)  dtype=uint8  min=0.000  max=1.000
64134.npy  shape=(1, 256, 256)  dtype=uint8  min=0.000  max=1.000
18929.npy  shape=(1, 256, 320)  dtype=uint8  min=0.000  max=1.000
36106.npy  shape=(1, 133, 609)  dtype=uint8  min=0.000  max=1.000
422.npy  shape=(1, 520, 696)  dtype=uint8  min=0.000  max=1.000


## Train Segmenter

In [30]:
# Data
seg_pairs = collect_pairs(AUTHENTIC_DIR, FORGED_DIR) 
seg_train, seg_val, seg_test = split_pairs(seg_pairs)
SEG_RESUME = True
seg_train_dl = DataLoader(
    SegDataset(seg_train, MASK_DIR, seg_transforms(True),  image_size=512),
    batch_size=SEG_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False) 
seg_val_dl   = DataLoader(
    SegDataset(seg_val,   MASK_DIR, seg_transforms(False), image_size=512),
    batch_size=SEG_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False)
seg_test_dl  = DataLoader(
    SegDataset(seg_test,  MASK_DIR, seg_transforms(False), image_size=512),
    batch_size=SEG_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False)

# Model
seg_model     = build_segmenter().to(DEVICE)
seg_criterion = BCEDiceLoss()
seg_optimizer = AdamW(seg_model.parameters(), lr=SEG_LR, weight_decay=SEG_WEIGHT_DECAY)
seg_scheduler = CosineAnnealingLR(seg_optimizer, T_max=SEG_NUM_EPOCHS, eta_min=1e-6)

start_epoch, best_val_dice, epochs_no_improve = 0, 0.0, 0
seg_train_losses, seg_val_losses, seg_val_dices = [], [], []

if SEG_RESUME and SEG_CHECKPOINT.exists():
    ckpt = torch.load(SEG_CHECKPOINT, map_location='cpu')
    seg_model.load_state_dict(ckpt['model_state'])
    seg_optimizer.load_state_dict(ckpt['optimizer_state'])
    seg_scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch    = ckpt['epoch']
    best_val_dice  = ckpt['best_val_dice']
    log.info(f'Resumed from epoch {start_epoch}  (best val Dice: {best_val_dice:.4f})')
elif SEG_RESUME:
    log.warning('SEG_RESUME=True but no checkpoint found. Training from scratch.')

# Training Loop
log.info('Starting segmenter training …')
for epoch in range(start_epoch, SEG_NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_m = seg_train_epoch(
        seg_model, seg_train_dl, seg_optimizer, seg_criterion, DEVICE, epoch, SEG_NUM_EPOCHS)
    val_loss, val_m = seg_evaluate(seg_model, seg_val_dl, seg_criterion, DEVICE)
    seg_scheduler.step()

    seg_train_losses.append(train_loss)
    seg_val_losses.append(val_loss)
    seg_val_dices.append(val_m['dice'])

    log.info(
        f'Epoch [{epoch+1:02d}/{SEG_NUM_EPOCHS}]  '
        f'train_loss={train_loss:.4f}  '
        f'val_loss={val_loss:.4f}  val_Dice={val_m["dice"]:.4f}  '
        f'val_IoU={val_m["iou"]:.4f}  val_Prec={val_m["precision"]:.4f}  '
        f'val_Rec={val_m["recall"]:.4f}  ({time.time()-t0:.1f}s)'
    )

    save_checkpoint({
        'epoch': epoch+1, 'model_state': seg_model.state_dict(),
        'optimizer_state': seg_optimizer.state_dict(),
        'scheduler_state': seg_scheduler.state_dict(),
        'best_val_dice': best_val_dice,
    }, SEG_CHECKPOINT)

    if val_m['dice'] > best_val_dice:
        best_val_dice, epochs_no_improve = val_m['dice'], 0
        torch.save(seg_model.state_dict(), SEG_BEST)
        log.info(f'  ↑ New best val Dice: {best_val_dice:.4f} → {SEG_BEST}')
    else:
        epochs_no_improve += 1
        log.info(f'  No improvement ({epochs_no_improve}/{SEG_PATIENCE})')

    if epochs_no_improve >= SEG_PATIENCE:
        log.info(f'Early stopping — no improvement for {SEG_PATIENCE} epochs')
        break

log.info('Segmenter training complete.')

2026-04-21 23:21:15,522  Authentic: 2377  |  Forged: 2751
2026-04-21 23:21:15,530  Train: 3589  Val: 769  Test: 770
2026-04-21 23:21:16,266  Resumed from epoch 21  (best val Dice: 0.4250)
2026-04-21 23:21:16,267  Starting segmenter training …
Epoch [22/30] Train: 100%|████████████████████████| 225/225 [06:30<00:00,  1.73s/it, loss=0.3837, dice=0.3961]
2026-04-21 23:28:27,672  Epoch [22/30]  train_loss=0.3611  val_loss=0.3726  val_Dice=0.4423  val_IoU=0.2954  val_Prec=0.4980  val_Rec=0.4109  (431.4s)
2026-04-21 23:28:28,270  Checkpoint saved → /kaggle/working/segmenter_checkpoint.pth
2026-04-21 23:28:28,459    ↑ New best val Dice: 0.4423 → /kaggle/working/segmenter_best.pth
Epoch [23/30] Train: 100%|████████████████████████| 225/225 [06:26<00:00,  1.72s/it, loss=0.2653, dice=0.6184]
2026-04-21 23:35:34,811  Epoch [23/30]  train_loss=0.3551  val_loss=0.3673  val_Dice=0.4505  val_IoU=0.3022  val_Prec=0.4932  val_Rec=0.4282  (426.4s)
2026-04-21 23:35:35,392  Checkpoint saved → /kaggle/work

## Segmenter Evaluation & Plots

In [32]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(seg_train_losses, label='Train Loss')
ax1.plot(seg_val_losses,   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Segmenter — Loss Curves'); ax1.legend()
ax2.plot(seg_val_dices, label='Val Dice', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
ax2.set_title('Segmenter — Validation Dice'); ax2.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segmenter_training_curves.png', dpi=150)
plt.show()

# Test evaluation
seg_model.load_state_dict(torch.load(SEG_BEST, map_location=DEVICE))
test_loss, test_m = seg_evaluate(seg_model, seg_test_dl, seg_criterion, DEVICE)

print('\n' + '='*60)
print('SEGMENTER — FINAL TEST SET RESULTS')
print('='*60)
for k, v in test_m.items():
    print(f'  {k:12s}: {v:.4f}')

# Prediction samples
seg_model.eval()
imgs_sample, masks_sample = next(iter(seg_test_dl))
imgs_sample = imgs_sample[:6].to(DEVICE)
with torch.no_grad():
    preds_sample = (torch.sigmoid(seg_model(imgs_sample)) >= 0.5).float().cpu()

fig, axes = plt.subplots(6, 3, figsize=(9, 18))
mean = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
for i in range(6):
    img_np = np.clip(imgs_sample[i].cpu().numpy().transpose(1,2,0) * std + mean, 0, 1)
    axes[i,0].imshow(img_np);                        axes[i,0].set_title('Image')
    axes[i,1].imshow(masks_sample[i,0], cmap='gray'); axes[i,1].set_title('Ground Truth')
    axes[i,2].imshow(preds_sample[i,0], cmap='gray'); axes[i,2].set_title('Prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segmenter_predictions.png', dpi=150)
plt.show()


SEGMENTER — FINAL TEST SET RESULTS
  dice        : 0.4457
  iou         : 0.2974
  precision   : 0.5193
  recall      : 0.4082


## Download Saved Models

In [ ]:
from IPython.display import FileLink, display
print('Output files:')
for f in sorted(OUTPUT_DIR.glob('*.pth')):
    print(f'  {f}')
    display(FileLink(str(f)))
for f in sorted(OUTPUT_DIR.glob('*.png')):
    print(f'  {f}')
    display(FileLink(str(f)))